# Alibi Explain Implementation
This notebook demonstrates Alibi Explain on the Medical Insurance dataset.

## Requirements
Run the following to install required dependencies. See `requirements-xai-frameworks.txt` for details.

In [ ]:
!pip install alibi[all] pandas numpy scikit-learn tensorflow>=2.10.0,<2.16.0

## Setup and Data
Loading the Medical Insurance dataset (dummy tabular data).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder

# Dummy data generator
np.random.seed(42)
df = pd.DataFrame({
    'age': np.random.randint(18, 65, 1000),
    'sex': np.random.choice(['male', 'female'], 1000),
    'bmi': np.random.uniform(18, 40, 1000),
    'children': np.random.randint(0, 5, 1000),
    'smoker': np.random.choice(['yes', 'no'], 1000),
    'region': np.random.choice(['southwest', 'southeast', 'northwest', 'northeast'], 1000),
    'charges': np.random.uniform(1000, 50000, 1000)
})

categorical_features = ['sex', 'smoker', 'region']
numeric_features = ['age', 'bmi', 'children']

# Alibi tabular explainers largely prefer Ordinal Encoded numeric mappings 
# rather than sparse One Hot Encoders
encoder = OrdinalEncoder()
df[categorical_features] = encoder.fit_transform(df[categorical_features])

X = df[categorical_features + numeric_features]
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
feature_names = list(X.columns)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
predict_fn = lambda x: model.predict(x)
print("Data loaded and model trained!")


## Explanability Framework API
Implementing XAI Platform internal API conventions for Alibi: `create_explainer`, `compute_global_alibi`, and `explain_instance`.

In [ ]:
from alibi.explainers import ALE, AnchorTabular

class AlibiServiceStub:
    @staticmethod
    def create_explainer(predict_fn, feature_names):
        # We will initialize ALE for global importance. 
        # (AnchorTabular is for local classification, so for regression we often use ALE / KernelShap)
        ale = ALE(predict_fn, feature_names=feature_names, target_names=['charges'])
        return ale
        
    @staticmethod
    def compute_global_alibi(explainer, X_bg):
        # Accumulated Local Effects (ALE)
        exp = explainer.explain(X_bg.values)
        return exp
        
    @staticmethod
    def explain_instance(predict_fn, X_train, feature_names, instance):
        # For local, we simulate a local explainer. Alibi's Anchor is for classification. 
        # Since this is regression, we might use normal SHAP/KernelShap from Alibi.
        from alibi.explainers import KernelShap
        explainer = KernelShap(predict_fn)
        explainer.fit(X_train.sample(50).values) # fit background
        explanation = explainer.explain(instance.values)
        return explanation

service = AlibiServiceStub()
explainer = service.create_explainer(predict_fn, feature_names)


## Global Explanation (ALE)

In [ ]:
global_exp = service.compute_global_alibi(explainer, X_train.sample(100))
print("ALE Explanations generated successfully. Plotting usually done via `from alibi.explainers.ale import plot_ale`")
print(f"Features explained: {global_exp.feature_names}")


## Local Explanation (KernelShap via Alibi)

In [ ]:
sample_instance = X_test.iloc[0:1]
local_exp = service.explain_instance(predict_fn, X_train, feature_names, sample_instance)
print("Local Explanation SHAP values (via Alibi):")
print(local_exp.shap_values)
